# Part 3 — Multi-Agent Supervisor

Revenue Agent + Expenditure Agent (`agents.py`) under a LangGraph supervisor (`part3_supervisor.py`,
`langgraph_supervisor.create_supervisor`). See `README.md` for the full writeup: architecture,
assumptions, and a real routing-quality finding from development (a demo query initially only
invoked one agent due to accidental context overlap between the two agents' page scoping).

## Sub-agents, verified individually before wiring the supervisor

Isolating failure points: confirm each agent answers correctly on its own before testing the
supervisor's routing on top of them.

In [1]:
from agents import build_revenue_agent, build_expenditure_agent

revenue_agent = build_revenue_agent()
r = revenue_agent.invoke({"messages": [{"role": "user", "content": "What is the largest single source of government revenue in FY2024, and what is its amount?"}]})
print("REVENUE AGENT (standalone):")
print(r["messages"][-1].content)

REVENUE AGENT (standalone):
Based on Table 2.1 in the FY2024 Budget, the largest single source of government revenue in FY2024 is **Corporate Income Tax at $28.03 billion**.

This is followed by Personal Income Tax at $18.07 billion and Goods and Services Tax at $19.39 billion. However, Corporate Income Tax remains the single largest revenue source for FY2024.


In [2]:
expenditure_agent = build_expenditure_agent()
r = expenditure_agent.invoke({"messages": [{"role": "user", "content": "How much is the Future Energy Fund being topped up by, and what will the money be used for?"}]})
print("EXPENDITURE AGENT (standalone):")
print(r["messages"][-1].content)

EXPENDITURE AGENT (standalone):
Based on the expenditure context, the **Future Energy Fund is being topped up by $5.0 billion**.

The money will be used to **invest in critical infrastructure for the energy transition**.


## Supervisor: assignment's exact required query

"What are the key government revenue streams, and how will the Budget for the Future Energy Fund
be supported?" — the trace below shows the supervisor's actual routing decisions, not an assumed
or asserted mechanism: question -> routes to revenue_agent -> tool call -> revenue synthesis ->
back to supervisor -> routes to expenditure_agent -> tool call -> expenditure synthesis -> back to
supervisor -> final comprehensive synthesis.

In [3]:
from part3_supervisor import build_supervisor, run_query, print_trace, DEMO_QUERIES

app = build_supervisor()
q1_label = "Q1 (assignment's exact query, dual-agent)"
answer, trace = run_query(app, DEMO_QUERIES[q1_label])
print("TRACE:")
print_trace(trace)
print("\nAGENTS INVOKED:", sorted(set(t["actor"] for t in trace) - {"user", "supervisor"}))
print("\nFINAL ANSWER:\n", answer)

TRACE:
[1] HumanMessage   actor=user                     What are the key government revenue streams, and how will the Budget for the Future Energy Fund be supported?
[2] AIMessage      actor=supervisor               thinking: {'signature': 'Et0CCpABCBEYAipA4XpmensSSHZGs4VLWAx4oq7gOynBSMXncSp6IGSpZ7LxX6nuBH5zUOvF9BqQ/FN/2isR1g8NYfPPtDiEZeHNUzIPY | tool_use: {'id': 'toolu_01Q6THX17467HrZZTQf8SsaS', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'transfer_to_revenue_agent',
[3] ToolMessage    actor=transfer_to_revenue_agent Successfully transferred to revenue_agent
[4] AIMessage      actor=revenue_agent            thinking: {'signature': 'EqMCCpABCBEYAipAqUraHBda6qDtYL6d5nIHVyshm1ya9Rz0JSIEgrtIq+Y+FU+13UZ9aUG8zrmZHQX1N7W8uL8Xay2WS6hs91rShzIPY | tool_use: {'id': 'toolu_01SJf9C9goupNZAM4bKg1pE6', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'revenue_context', 'type': '
[5] ToolMessage    actor=revenue_context          --- PAGE 5 ---
MINISTRY OF FINANCE 
 
5 
 
01 Update on Financ

## Verification against ground truth

- Future Energy Fund must be stated as **$5.0 billion**, with the "critical infrastructure for the
  energy transition" purpose from page 18 — not just a bare number.
- Both `revenue_agent` and `expenditure_agent` must appear in the trace's actor set (verified from
  the trace structure itself, not inferred from the answer sounding plausible).
- Revenue streams named in the answer must match the real 12-item Operating Revenue tax list from
  Part 1 (no invented categories).

In [4]:
actors = set(t["actor"] for t in trace)
checks = {
    "Future Energy Fund states $5.0 billion": "5.0 billion" in answer or "$5.0" in answer,
    "Future Energy Fund purpose (energy transition) present": "energy transition" in answer.lower(),
    "revenue_agent invoked (from trace)": "revenue_agent" in actors,
    "expenditure_agent invoked (from trace)": "expenditure_agent" in actors,
}
for label, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {label}")

PASS  Future Energy Fund states $5.0 billion
PASS  Future Energy Fund purpose (energy transition) present
PASS  revenue_agent invoked (from trace)
PASS  expenditure_agent invoked (from trace)


## Various queries demonstrating collaborative routing

Four queries designed to prove the supervisor genuinely **routes** (not reflexively calling both
agents every time) as well as genuinely **collaborates** when a query needs both:
1. The assignment's exact query (dual-agent) — run above.
2. Revenue-only query — should invoke only `revenue_agent`.
3. Expenditure-only query — should invoke only `expenditure_agent`.
4. A second, differently-phrased dual-agent query — should invoke both again, confirming query 1
   wasn't a one-off fluke.

In [5]:
import json

all_results = {q1_label: {"query": DEMO_QUERIES[q1_label], "trace": trace, "final_answer": answer}}

for label, query in DEMO_QUERIES.items():
    if label == q1_label:
        continue
    print(f"\n{'=' * 80}\n{label}\nQuery: {query}\n{'=' * 80}")
    a, t = run_query(app, query)
    print("\nAGENTS INVOKED:", sorted(set(x["actor"] for x in t) - {"user", "supervisor"}))
    print("\nFINAL ANSWER:\n", a)
    all_results[label] = {"query": query, "trace": t, "final_answer": a}

with open("trace.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)
print("\nWrote trace.json with all 4 demo queries.")


Q2 (revenue-only, tests selective routing)
Query: What is the largest single source of government revenue?



AGENTS INVOKED: ['revenue_agent', 'revenue_context', 'transfer_back_to_supervisor', 'transfer_to_revenue_agent']

FINAL ANSWER:
 The largest single source of government revenue is **Corporate Income Tax**.

- In **Revised FY2023**, Corporate Income Tax collections totaled **$28.4 billion**, representing **27.2%** of Operating Revenue — the single largest component of government revenue, ahead of all other tax and non-tax sources. This figure was $4.1 billion (17.0%) higher than originally estimated, driven by stronger-than-expected economic growth in 2022.
- For **Estimated FY2024**, Corporate Income Tax is projected at **$28.03 billion**, a slight decline of $0.35 billion (-1.2%) from the revised FY2023 figure, but it is still expected to remain the top revenue source.

For context, the next largest revenue sources in FY2023 were:
- **Personal Income Tax**: $17.5 billion (16.8% of revenue)
- **Goods and Services Tax (GST)**: $16.4 billion (15.7% of revenue)
- **Other Taxes** (Foreign


AGENTS INVOKED: ['expenditure_agent', 'expenditure_context', 'transfer_back_to_supervisor', 'transfer_to_expenditure_agent']

FINAL ANSWER:
 **Answer: GST Voucher Fund Top-up**

**Amount:** $6.0 billion (S$6,000 million) — the largest single fund top-up announced in Budget 2024.

**Reason/Purpose:** The top-up is intended to **meet the steady-state cashflow needs of the enhanced, now-permanent GST Voucher scheme**. In other words, since the GST Voucher scheme has been made a permanent feature (rather than a one-off/temporary support measure) and has been enhanced, the government is injecting funds now to ensure the Fund has enough resources to sustain payouts to eligible households on an ongoing basis into the future.

This $6.0 billion top-up is part of a larger $20.4 billion (S$20,352 million) total committed to Top-ups to Endowment and Trust Funds in Budget 2024, which also includes top-ups to funds such as the Future Energy Fund ($5.0 billion), Edusave Endowment Fund ($2.0 billion


AGENTS INVOKED: ['expenditure_agent', 'expenditure_context', 'revenue_agent', 'revenue_context', 'transfer_back_to_supervisor', 'transfer_to_expenditure_agent', 'transfer_to_revenue_agent']

FINAL ANSWER:
 ## Summary

**1. Largest source of tax revenue: Corporate Income Tax**
- Corporate Income Tax is the single largest tax revenue source, comprising **27.2%** of Operating Revenue in FY2023 — ahead of Personal Income Tax (16.8%) and Goods and Services Tax (15.7%).
- In dollar terms, collections were **$28.38 billion** (Revised FY2023) and are estimated at **$28.03 billion** (Estimated FY2024).

**2. Future Energy Fund infrastructure spending purpose**
According to the budget document, the Government is establishing the Future Energy Fund with an initial injection of **$5.0 billion** in FY2024 specifically to **"invest in critical infrastructure for the energy transition."** This is a newly created fund (no allocation existed in FY2023), and it represents one component of the $20.4 bil

## Routing-pattern summary

Expected pattern: dual / revenue-only / expenditure-only / dual. Confirms the supervisor's
routing decisions are genuine and query-dependent, not hardcoded or reflexive.

In [6]:
for label, result in all_results.items():
    agents_used = sorted(set(t["actor"] for t in result["trace"]) & {"revenue_agent", "expenditure_agent"})
    print(f"{label}: {agents_used}")

Q1 (assignment's exact query, dual-agent): ['expenditure_agent', 'revenue_agent']
Q2 (revenue-only, tests selective routing): ['revenue_agent']
Q3 (expenditure-only, tests selective routing): ['expenditure_agent']
Q4 (second dual-agent query, different phrasing): ['expenditure_agent', 'revenue_agent']
